In [24]:
import numpy as np
from numba import njit, experimental, int32
import numba as nb

In [98]:


spec = [
    ("batchsize", nb.int32),
    ("windowsize", nb.int32),
    ("lookforward", nb.int32),
    ("indexes", nb.int32[:]),
    # indexes where the pointers are
    ("tensors", nb.float32[:,:,:]),
    # data from tfdata, [timeframe, feature, entries]
    ("lookforwardoutput", nb.float32[:,:,:]),
    # lookforwardoutput, [batch, current + lookforward, labels]
    ("dataoutput", nb.float32[:,:,:,:]),
    # data output, [batch, timeframe, feature, entries]
    ("deltas", nb.float32[:]),
    # approximate time deltas for each timeframe
    ("currenttimes", nb.int32[:]),
]

# Features are as follows: [timeframe, open, high, low, close, volume]

@experimental.jitclass(spec)
class slidecontainer:
    def __init__(self, tensors, batchsize, windowsize, lookforward):
        self.tensors = tensors
        """Indexed by increasing timeframe, base is at index 0 -> [timeframe, feature, entry]"""
        self.batchsize = batchsize
        self.windowsize = windowsize
        self.lookforward = lookforward
        self.indexes = np.zeros(shape=(tensors.shape[0]), dtype=np.int32)
        """Indexed by increasing timeframe, base is at index 0"""

        # Set up output buffers
        self.lookforwardoutput = np.zeros(shape=(self.batchsize, self.lookforward +1, 2), dtype=np.float32)
        self.dataoutput = np.zeros(shape=(self.batchsize, tensors.shape[0], tensors.shape[1], self.windowsize), dtype=np.float32)

        # Compute time variables
        self.deltas = np.zeros(shape=(tensors.shape[0]), dtype=np.float32)
        """Approximate delta t between timeframe entries"""
        self.currenttimes = np.zeros(shape=(tensors.shape[0]), dtype=np.int32)
        """Timestamp at the indexes for each timeframe"""

        # Update timestamps
        self.updateCurrentTimestamps()

        # Compute deltas
        for i in range(tensors.shape[0]):
            self.deltas[i] = self.tensors[i,0,1] - self.tensors[i,0,1] # get delta


    def updateCurrentTimestamps(self):
        for i in range(self.currenttimes.shape[0]):
            self.currenttimes[i] = self.tensors[i,0,self.indexes[i]] # get timestamp

    def getLatestTime(self) -> np.float32:

        self.updateCurrentTimestamps()

        latest = 0
        for timestamp in self.currenttimes:
            thislatest = timestamp
            if thislatest > latest:
                latest = thislatest

        return latest

    def stepToPresent(self, timestamp = None):

        if timestamp is None:
            self.updateCurrentTimestamps()
            timestamp = self.currenttimes[0]

        for timeframe, timeframetensor in enumerate(self.tensors):
            while timestamp > timeframetensor[0, self.indexes[timeframe] - self.lookforward]:
                self.indexes[timeframe] += 1

    def init_from_zero(self):
        # Slide all indexes to the minimum window size
        self.indexes[:] = self.windowsize + self.lookforward

        latest = self.getLatestTime()
        self.stepToPresent(latest)

    def fillresponsearrays(self, batchindex):

        # iterate over each timeframe
        for timeframe, timeframetensor in enumerate(self.tensors):
            # We are skipping the timestamp at index 0
            # We are only selecting the window size
            windowstart = self.indexes[timeframe] - self.lookforward - self.windowsize
            windowend = self.indexes[timeframe] - self.lookforward

            # fill data output
            self.dataoutput[batchindex][timeframe] = timeframetensor[1:, windowstart: windowend]

            # fill lookforward
            self.lookforwardoutput[batchindex][timeframe] = timeframetensor[1:, windowend: self.indexes[timeframe]]


    def next(self):

        # fill up batches
        for i in range(self.batchsize):
            # Step up
            self.indexes[0] += 1
            self.stepToPresent(self.indexes[0])
            # Fill
            self.fillresponsearrays(i)

        return self.dataoutput, self.lookforwardoutput


In [99]:
sc = slidecontainer(np.zeros(shape=(5, 5, 10000), dtype=np.float32), np.int32(50), np.int32(100), np.int32(5))
sc.init_from_zero()
print(sc.deltas)

[0. 0. 0. 0. 0.]


In [ ]:
import TensorSlider

slider = TensorSlider.WindowSlider()